# 05 Compare Versions

Compare saved JSON snapshots from `outputs/version_snapshots`. This helps separate changes caused by config/logic from changes caused by input data.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd
while project_root.name != "SEN05" and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / "backtest_optimize"
SNAPSHOT_DIR = BACKTEST_ROOT / "outputs" / "version_snapshots"

In [ ]:
import json
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [ ]:
files = sorted(SNAPSHOT_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
records = []
for path in files:
    with path.open("r", encoding="utf-8") as handle:
        record = json.load(handle)
    record["snapshot_path"] = str(path)
    records.append(record)

if not records:
    raise FileNotFoundError("No snapshots found. Run notebook 01 and save a snapshot first.")

df = pd.json_normalize(records, sep=".")
core_cols = [
    "created_at",
    "name",
    "git_commit",
    "signal_file",
    "signal_file_md5",
    "config_hash",
    "snapshot_hash",
    "result_summary.signal_count",
    "result_summary.accepted_count",
    "result_summary.expectancy_r",
    "result_summary.skip_rate",
    "result_summary.ambiguity_rate",
    "result_summary.mean_bar_mae_r",
    "result_summary.mean_bar_mfe_r",
]
available = [col for col in core_cols if col in df.columns]
display(df[available].head(30))

In [ ]:
# Pick two rows by index to inspect differences.
LEFT_INDEX = 0
RIGHT_INDEX = 1 if len(df) > 1 else 0

left = df.iloc[LEFT_INDEX]
right = df.iloc[RIGHT_INDEX]

diff_rows = []
for col in sorted(df.columns):
    left_value = left.get(col)
    right_value = right.get(col)
    if str(left_value) != str(right_value):
        diff_rows.append({"field": col, "left": left_value, "right": right_value})

diff = pd.DataFrame(diff_rows)
display(diff.head(100))